In [1]:
import duckdb #version 1.1.3
import geopandas as gpd #version 0.14.1
from shapely import wkt
from shapely.geometry import shape
from threading import Thread, current_thread

In [2]:
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("azure")
con.load_extension("azure")
con.install_extension("json")
con.load_extension("json")
print(duckdb.__version__) #previously 0.9.2

1.1.3


##### Set parameters/variables

In [3]:
pid_fld = 'dgr_blk'
nwiurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_wetlands.parquet"
wetattrfld = 'CLASS_NAME'
hucsurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_wsheds.parquet"
hucidfld = 'WATERSHED_CODE'
crossWalk_json = 'https://giscog.blob.core.windows.net/abdu/caWetlands.json'
fowlDemo = r'azure://abdu/WaterfowlDemographic.parquet'
nrgy_csv = 'azure://abdu/ehjv_kcal.csv'
protLands = r'D:\ABDUBounds\ABDU_Canada_Data\parquet\ProtectedConservedArea_2024.parquet'
urbanMask = r"D:\ABDUBounds\abdu_ca_urban.parquet"
demand = 'azure://abdu/Demand9Species_Merged.parquet'
politicalbndry = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\ca_prov.parquet"

In [4]:
in_crs = {p:'ESRI:102008' for p in (nwiurl,hucsurl,protLands) if not p is None}

In [5]:
wet_flds = [wetattrfld, hucidfld]

In [6]:
params = (nwiurl, politicalbndry, hucsurl, fowlDemo, protLands, urbanMask, demand, crossWalk_json, nrgy_csv)
if any([i in '_'.join(params) for i in ('azure','giscog')]):
    con.sql("SET azure_transport_option_type = 'curl'")
    con.sql('''CREATE OR REPLACE SECRET secret0 (TYPE AZURE, ACCOUNT_NAME 'giscog')''')

In [7]:
param_crs = {p:'EPSG:5070' for p in params[:-2]}
if len(in_crs)>0:
    for k, v in in_crs.items():
            param_crs[k] = v
param_crs 

{'D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_wetlands.parquet': 'ESRI:102008',
 'D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\ca_prov.parquet': 'EPSG:5070',
 'D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_wsheds.parquet': 'ESRI:102008',
 'azure://abdu/WaterfowlDemographic.parquet': 'EPSG:5070',
 'D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\ProtectedConservedArea_2024.parquet': 'ESRI:102008',
 'D:\\ABDUBounds\\abdu_ca_urban.parquet': 'EPSG:5070',
 'azure://abdu/Demand9Species_Merged.parquet': 'EPSG:5070'}

In [8]:
### transform/reproject geometry when necessary
def tfrm_str(geom='geometry', in_crs='EPSG:5070', out_crs=param_crs[hucsurl]):
    """
    format str for sql query to reproject geometry of duckdb table
    """
    if not in_crs==out_crs:
        s = f"ST_Transform({geom}, '{in_crs}', '{out_crs}')"
    else:
        s = geom
    return s
### Should add check for linear unit for hectare calculations

In [9]:
def sql_picker(huc, pqt_url, water=True):
    if pqt_url == nwiurl:
        if '**' in nwiurl:
            sql = '''
            SELECT {2}, ST_AsWKB(ST_Intersection(ST_GeomFromWKB(wetlnd.geometry), ST_GeomFromWKB(hucs.geometry)) AS geometry
            FROM (SELECT {3}, geometry FROM read_parquet('{1}',hive_partitioning=true) 
            WHERE {4} = '{0}' AND NOT ({3} LIKE 'R%UB%' OR {3} LIKE 'R%SB%' OR {3} LIKE 'R%RB%')) AS wetlnd
            JOIN hucs ON 
            ST_Intersects(ST_GeomFromWKB(wetlnd.geometry), ST_GeomFromWKB(hucs.geometry)))
            '''.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld)
        else:
            if water:
                sql = """
                SELECT {2}, ST_AsText(ST_Intersection({5}, hucs.geometry)) AS geometry
                FROM read_parquet('{1}') wet, hucs
                WHERE hucs.{4} = '{0}'
                AND ST_Intersects({5}, hucs.geometry)
                AND NOT ST_IsEMpty(ST_Intersection({5}, hucs.geometry))
                """.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld, tfrm_str('wet.geometry', param_crs[nwiurl]))             
            else:
                sql = '''
                    SELECT {2}, ST_AsText(ST_Intersection({5}, hucs.geometry)) AS geometry
                    FROM read_parquet('{1}') wet, hucs
                    WHERE hucs.{4} = '{0}'
                    AND wet.{3} != 'Open Water'
                    AND ST_Intersects({5}, hucs.geometry)
                    AND NOT ST_IsEMpty(ST_Intersection({5}, hucs.geometry))
                    '''.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld, tfrm_str('wet.geometry', param_crs[nwiurl]))
    elif pqt_url == protLands and '**' in protLands:
            sql = """
            SELECT CATEGORY, hucs, huc2, huc4, ST_AsText(ST_Intersection(ST_GeomFromWKB(hucs.geometry), ST_GeomFromWKB(prot.geometry)) AS geometry
            FROM (SELECT CATEGORY, geometry FROM read_parquet('{1}', hive_partitioning=true)
            WHERE CATEGORY IN ('Fee', 'Easements', 'Other') AND huc4 = '{0}') AS prot
            JOIN hucs ON 
            ST_Intersects(ST_GeomFromWKB(hucs.geometry), ST_GeomFromWKB(prot.geometry))
            """.format(huc, protLands)
    else:
        sql = """
        SELECT {1}, pqt.* EXCLUDE geometry, ST_AsText(ST_Intersection(hucs.geometry, {2})) AS geometry
        FROM read_parquet('{0}') AS pqt
        JOIN hucs ON 
        ST_Intersects(hucs.geometry, {2})
        WHERE hucs.{1} = '{3}'
        """.format(pqt_url, hucidfld, tfrm_str('pqt.geometry', param_crs[pqt_url]), huc)
    return sql

In [10]:
def write_from_thread(con, url, table):
    local_con = con.cursor()
    huc = str(current_thread().name)
    if any([i in url for i in ('azure','giscog')]):
        local_con.sql('''CREATE OR REPLACE SECRET secret0 (TYPE AZURE, ACCOUNT_NAME 'giscog')''')
        local_con.sql("SET azure_transport_option_type = 'curl'")
    sql = sql_picker(huc, url)
    rows = local_con.sql(sql).fetchall()
    if len(rows)>0:
        n = local_con.sql(f"DESCRIBE {table}").df().column_name.shape[0]
        sql = f'''INSERT INTO {table}
        VALUES ({', '.join(['?' for i in range(n)])})
        '''
        result = local_con.executemany(sql, rows).fetchall()

In [11]:
sql = f"""CREATE OR REPLACE TABLE cn_prov AS
SELECT * FROM read_parquet('{politicalbndry}')
"""
con.execute(sql)
con.sql('DESCRIBE cn_prov')

┌───────────────┬────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │                      column_type                       │  null   │   key   │ default │  extra  │
│    varchar    │                        varchar                         │ varchar │ varchar │ varchar │ varchar │
├───────────────┼────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ FID           │ BIGINT                                                 │ YES     │ NULL    │ NULL    │ NULL    │
│ postal        │ VARCHAR                                                │ YES     │ NULL    │ NULL    │ NULL    │
│ Nom_Fr        │ VARCHAR                                                │ YES     │ NULL    │ NULL    │ NULL    │
│ Name_EN       │ VARCHAR                                                │ YES     │ NULL    │ NULL    │ NULL    │
│ Shape_Leng    │ DOUBLE                                                 │ YES  

##### Join Demand with province

In [12]:
sql = f"""CREATE OR REPLACE TABLE demand AS
SELECT * FROM read_parquet('{demand}')
"""
con.execute(sql)
con.sql('DESCRIBE demand')

┌─────────────┬────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │                        column_type                         │  null   │   key   │ default │  extra  │
│   varchar   │                          varchar                           │ varchar │ varchar │ varchar │ varchar │
├─────────────┼────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID    │ BIGINT                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ species     │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ fips        │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ CODE        │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ LTADUD      │ INTEGER                                         

In [13]:
con.sql("SELECT * FROM cn_prov LIMIT 1")
con.sql("SELECT ST_Extent(ST_Extent_Agg(geometry)) AS box2d FROM cn_prov")

┌───────────────────────────────────────────────┐
│                     box2d                     │
│                    box_2d                     │
├───────────────────────────────────────────────┤
│ BOX(-2665004.5 2151070.5, 3188701.25 6018797) │
└───────────────────────────────────────────────┘

In [14]:
con.sql("SELECT * FROM demand LIMIT 1")
con.sql("SELECT ST_Extent(ST_Extent_Agg(geometry)) AS box2d FROM demand")

┌────────────────────────────────────────────────────┐
│                       box2d                        │
│                       box_2d                       │
├────────────────────────────────────────────────────┤
│ BOX(-3236326.25 -848418.6875, 3188171.5 5361325.5) │
└────────────────────────────────────────────────────┘

In [13]:
con.sql("""
    CREATE OR REPLACE TABLE provXdemand AS
    SELECT FID, postal, Nom_Fr, Name_EN, OBJECTID, species, fips, CODE, LTADUD, X80DUD, LTAPopObj, X80PopObj, LTADemand, X80Demand,
    ST_Intersection(cn_prov.geometry, demand.geometry) AS geometry
    FROM cn_prov, demand
 """)
con.sql("DESCRIBE provXdemand")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ FID         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ postal      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Nom_Fr      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Name_EN     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OBJECTID    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ species     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fips        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ CODE        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LTADUD      │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ X80DUD      │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ LTAPopObj   │ INTE

In [14]:
con.sql("SELECT count(FID), sum(ST_Area(geometry)) FROM cn_prov"), con.sql("SELECT count(OBJECTID), sum(ST_Area(geometry)) FROM demand")

(┌────────────┬────────────────────────┐
 │ count(FID) │ sum(st_area(geometry)) │
 │   int64    │         double         │
 ├────────────┼────────────────────────┤
 │         13 │       9946176634441.87 │
 └────────────┴────────────────────────┘,
 ┌─────────────────┬────────────────────────┐
 │ count(OBJECTID) │ sum(st_area(geometry)) │
 │      int64      │         double         │
 ├─────────────────┼────────────────────────┤
 │           22328 │      73129204960930.75 │
 └─────────────────┴────────────────────────┘)

In [15]:
con.sql("SELECT count(FID), count(OBJECTID), count(distinct(FID)), count(distinct(OBJECTID)), sum(ST_Area(geometry)) FROM provXdemand")

┌────────────┬─────────────────┬─────────────────────┬──────────────────────────┬────────────────────────┐
│ count(FID) │ count(OBJECTID) │ count(DISTINCT FID) │ count(DISTINCT OBJECTID) │ sum(st_area(geometry)) │
│   int64    │      int64      │        int64        │          int64           │         double         │
├────────────┼─────────────────┼─────────────────────┼──────────────────────────┼────────────────────────┤
│     290264 │          290264 │                  13 │                    22328 │      11543129385136.25 │
└────────────┴─────────────────┴─────────────────────┴──────────────────────────┴────────────────────────┘

In [16]:
con.sql("SELECT min(ST_Area(geometry)), max(ST_Area(geometry)) FROM provXdemand")

┌────────────────────────┬────────────────────────┐
│ min(st_area(geometry)) │ max(st_area(geometry)) │
│         double         │         double         │
├────────────────────────┼────────────────────────┤
│                    0.0 │      8985317281.311028 │
└────────────────────────┴────────────────────────┘

In [17]:
(
    con.sql("SELECT count(*) FROM provXdemand WHERE ST_Area(geometry) = 0").fetchall(),
	con.sql("SELECT count(*) FROM provXdemand WHERE NOT ST_IsValid(geometry)").fetchall(),
	con.sql("SELECT count(*) FROM provXdemand WHERE ST_Extent(geometry) IS NULL").fetchall()
)

([(287765,)], [(0,)], [(287765,)])

In [18]:
con.sql("SELECT * FROM provXdemand WHERE FID IS NULL"), con.sql("SELECT * FROM provXdemand WHERE OBJECTID IS NULL")

(┌───────┬─────────┬─────────┬─────────┬──────────┬─────────┬─────────┬─────────┬────────┬────────┬───────────┬───────────┬───────────┬───────────┬──────────┐
 │  FID  │ postal  │ Nom_Fr  │ Name_EN │ OBJECTID │ species │  fips   │  CODE   │ LTADUD │ X80DUD │ LTAPopObj │ X80PopObj │ LTADemand │ X80Demand │ geometry │
 │ int64 │ varchar │ varchar │ varchar │  int64   │ varchar │ varchar │ varchar │ int32  │ int32  │   int32   │   int32   │  double   │  double   │ geometry │
 ├───────┴─────────┴─────────┴─────────┴──────────┴─────────┴─────────┴─────────┴────────┴────────┴───────────┴───────────┴───────────┴───────────┴──────────┤
 │                                                                          0 rows                                                                           │
 └───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘,
 ┌───────┬─────────┬─────────┬─────────┬─────

In [19]:
con.sql("""
	CREATE OR REPLACE TABLE provDemand_pivot AS
    PIVOT (
        SELECT * FROM provXdemand
		WHERE ST_Area(geometry) > 0
        AND fips LIKE 'n%'
        )
    ON Name_EN 
	USING sum(ST_Area(geometry))
	GROUP BY fips      
""")
con.sql("DESCRIBE provDemand_pivot")

┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ fips                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Alberta                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ British Columbia        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Manitoba                │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ New Brunswick           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Newfoundland & Labrador │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Northwest Territories   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Nova Scotia             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Ontario       

In [20]:
con.sql("SELECT * FROM provDemand_pivot LIMIT 1")

┌─────────────┬──────────────────┬───────────────────┬──────────┬───────────────┬─────────────────────────┬───────────────────────┬─────────────┬─────────┬──────────────────────┬────────┬──────────────┬────────┐
│    fips     │     Alberta      │ British Columbia  │ Manitoba │ New Brunswick │ Newfoundland & Labrador │ Northwest Territories │ Nova Scotia │ Ontario │ Prince Edward Island │ Québec │ Saskatchewan │ Yukon  │
│   varchar   │      double      │      double       │  double  │    double     │         double          │        double         │   double    │ double  │        double        │ double │    double    │ double │
├─────────────┼──────────────────┼───────────────────┼──────────┼───────────────┼─────────────────────────┼───────────────────────┼─────────────┼─────────┼──────────────────────┼────────┼──────────────┼────────┤
│ n119.5_56.5 │ 20571037658.2417 │ 892178.7033420503 │     NULL │          NULL │                    NULL │                  NULL │        NULL │    NUL

In [21]:
con.sql("""
        UNPIVOT provDemand_pivot
        ON * EXCLUDE fips
        INTO
        	NAME Name_EN
        	VALUE AREA
        ORDER BY fips
""")

┌─────────────┬──────────────┬────────────────────┐
│    fips     │   Name_EN    │        AREA        │
│   varchar   │   varchar    │       double       │
├─────────────┼──────────────┼────────────────────┤
│ n100.5_48.5 │ Manitoba     │ 19357291.718460508 │
│ n100.5_49.5 │ Manitoba     │  72504870926.52199 │
│ n100.5_50.5 │ Manitoba     │  71028658955.24568 │
│ n100.5_51.5 │ Manitoba     │   45147512649.0376 │
│ n100.5_52.5 │ Manitoba     │  28413105672.66545 │
│ n100.5_53.5 │ Manitoba     │  7966823611.851374 │
│ n101.5_49.5 │ Manitoba     │ 29843135683.529133 │
│ n101.5_49.5 │ Saskatchewan │ 42661735112.587234 │
│ n101.5_50.5 │ Manitoba     │  31877619268.68009 │
│ n101.5_50.5 │ Saskatchewan │  31258966412.28589 │
│     ·       │    ·         │          ·         │
│     ·       │    ·         │          ·         │
│     ·       │    ·         │          ·         │
│ n98.5_49.5  │ Manitoba     │  72533626336.28865 │
│ n98.5_50.5  │ Manitoba     │  46619938551.40249 │
│ n98.5_51.5

In [22]:
con.sql("SELECT count(distinct(fips)) FROM demand, cn_prov WHERE ST_Intersects(demand.geometry, cn_prov.geometry) and fips LIKE 'n%'")

┌──────────────────────┐
│ count(DISTINCT fips) │
│        int64         │
├──────────────────────┤
│                  319 │
└──────────────────────┘

In [23]:
con.sql("""
    CREATE OR REPLACE TABLE maxprov AS
    WITH tbl AS (
        UNPIVOT provDemand_pivot
        ON * EXCLUDE fips
        INTO
        	NAME Name_EN
        	VALUE AREA
        ORDER BY fips
    )
    SELECT * FROM (
        SELECT tbl.*, ROW_NUMBER() OVER (
        	PARTITION BY fips
        	ORDER BY AREA DESC, Name_EN ASC
        ) AS rn
        FROM tbl
    )
	WHERE rn=1
    ORDER BY fips
""")
con.sql("SELECT * FROM maxprov")

┌─────────────┬──────────────┬────────────────────┬───────┐
│    fips     │   Name_EN    │        AREA        │  rn   │
│   varchar   │   varchar    │       double       │ int64 │
├─────────────┼──────────────┼────────────────────┼───────┤
│ n100.5_48.5 │ Manitoba     │ 19357291.718460508 │     1 │
│ n100.5_49.5 │ Manitoba     │  72504870926.52199 │     1 │
│ n100.5_50.5 │ Manitoba     │  71028658955.24568 │     1 │
│ n100.5_51.5 │ Manitoba     │   45147512649.0376 │     1 │
│ n100.5_52.5 │ Manitoba     │  28413105672.66545 │     1 │
│ n100.5_53.5 │ Manitoba     │  7966823611.851374 │     1 │
│ n101.5_49.5 │ Saskatchewan │ 42661735112.587234 │     1 │
│ n101.5_50.5 │ Manitoba     │  31877619268.68009 │     1 │
│ n101.5_51.5 │ Manitoba     │   27728168157.2404 │     1 │
│ n101.5_53.5 │ Manitoba     │  40023529933.83826 │     1 │
│     ·       │    ·         │          ·         │     · │
│     ·       │    ·         │          ·         │     · │
│     ·       │    ·         │          

In [24]:
con.sql("""
    CREATE OR REPLACE TABLE demand AS
	SELECT demand.*, Name_EN FROM demand
    JOIN maxprov
        ON demand.fips=maxprov.fips
    WHERE Name_EN IS NOT NULL
    ORDER BY demand.fips
""")
con.sql("SELECT * EXCLUDE geometry FROM demand")

┌──────────┬─────────┬─────────────┬─────────┬─────────┬─────────┬───────────┬───────────┬───────────────────┬────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┐
│ OBJECTID │ species │    fips     │  CODE   │ LTADUD  │ X80DUD  │ LTAPopObj │ X80PopObj │     LTADemand     │     X80Demand      │                                                       bbox                                                        │ Name_EN  │
│  int64   │ varchar │   varchar   │ varchar │  int32  │  int32  │   int32   │   int32   │      double       │       double       │                            struct(xmin double, ymin double, xmax double, ymax double)                             │ varchar  │
├──────────┼─────────┼─────────────┼─────────┼─────────┼─────────┼───────────┼───────────┼───────────────────┼────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────

In [25]:
con.sql("SELECT distinct(Name_EN) FROM demand"), con.sql("SELECT count(*) FROM demand WHERE Name_EN IS NULL")

(┌─────────────────────────┐
 │         Name_EN         │
 │         varchar         │
 ├─────────────────────────┤
 │ Yukon                   │
 │ Nova Scotia             │
 │ Alberta                 │
 │ New Brunswick           │
 │ Ontario                 │
 │ Manitoba                │
 │ Newfoundland & Labrador │
 │ Saskatchewan            │
 │ Northwest Territories   │
 │ Prince Edward Island    │
 │ British Columbia        │
 │ Québec                  │
 ├─────────────────────────┤
 │         12 rows         │
 └─────────────────────────┘,
 ┌──────────────┐
 │ count_star() │
 │    int64     │
 ├──────────────┤
 │            0 │
 └──────────────┘)

In [26]:
# out_f = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_Demand.parquet"
# # con.sql(f"COPY (SELECT * FROM demand) TO '{out_f}' (FORMAT PARQUET)")
# sql = f"""CREATE OR REPLACE TABLE demand AS
# SELECT * FROM read_parquet('{out_f}')
# """
# con.execute(sql)
con.sql('DESCRIBE demand')

┌─────────────┬────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │                        column_type                         │  null   │   key   │ default │  extra  │
│   varchar   │                          varchar                           │ varchar │ varchar │ varchar │ varchar │
├─────────────┼────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID    │ BIGINT                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ species     │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ fips        │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ CODE        │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ LTADUD      │ INTEGER                                         

In [27]:
# con.sql("SELECT distinct(CODE) FROM demand")
con.sql("SELECT code, count(CODE) FROM demand GROUP BY CODE")

┌─────────┬─────────────┐
│  CODE   │ count(CODE) │
│ varchar │    int64    │
├─────────┼─────────────┤
│ 4D      │          42 │
│ 4B      │        1745 │
└─────────┴─────────────┘

In [28]:
spp_list = sorted([i[0] for i in con.sql(f"SELECT distinct(species) FROM demand").fetchall()])
spp_list

['ABDU', 'AGWT', 'AMWI', 'All', 'BWTE', 'GADW', 'MALL', 'NOPI', 'NSHO', 'WODU']

In [29]:
# con.sql("DROP TABLE provDemand_pivot")
con.sql("SHOW TABLES")

┌──────────────────┐
│       name       │
│     varchar      │
├──────────────────┤
│ cn_prov          │
│ demand           │
│ maxprov          │
│ provDemand_pivot │
│ provXdemand      │
└──────────────────┘

#### Join watersheds with province

In [33]:
sql = f"""CREATE OR REPLACE TABLE hucs AS
SELECT * FROM read_parquet('{hucsurl}')

"""
con.execute(sql)
# con.sql("DESCRIBE hucs")
# print(con.sql("SELECT count(*) FROM hucs").fetchone()[0])
con.sql("SELECT * FROM hucs")

┌──────────┬────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [34]:
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
    WITH code_map(code, name) AS (
        VALUES
            ('GS', 'Gulf of St Lawrence'),
            ('BF', 'Bay of Fundy')
    )
    SELECT
        h.OBJECTID,
        h.WATERSHED_CODE,
        h.MERGE_SRC,
        cm.name AS PROV,
        h.geometry
    FROM hucs AS h
    LEFT JOIN code_map AS cm
      ON substr(h.WATERSHED_CODE, 1, 2) = cm.code
""")
con.sql("SELECT * FROM hucs")

┌──────────┬────────────────┬──────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [35]:
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
	SELECT hucs.*, Name_EN
	FROM hucs
	LEFT JOIN cn_prov 
	ON ST_Within(ST_Centroid(hucs.geometry), ST_Transform(cn_prov.geometry, 'EPSG:5070', 'ESRI:102008'))
""")
con.sql("SELECT * FROM hucs")

┌──────────┬────────────────┬──────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [36]:
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
	SELECT OBJECTID, WATERSHED_CODE, MERGE_SRC, coalesce(PROV, Name_EN) AS PROV, geometry
	FROM hucs
""")
con.sql("SELECT * FROM hucs")

┌──────────┬────────────────┬──────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [37]:
con.sql("SELECT distinct(PROV), count(*) FROM hucs GROUP BY PROV ORDER BY PROV")

┌──────────────────────┬──────────────┐
│         PROV         │ count_star() │
│       varchar        │    int64     │
├──────────────────────┼──────────────┤
│ Bay of Fundy         │           11 │
│ Gulf of St Lawrence  │            9 │
│ New Brunswick        │          409 │
│ Nova Scotia          │          312 │
│ Prince Edward Island │           70 │
│ Québec               │         1545 │
│ NULL                 │           26 │
└──────────────────────┴──────────────┘

In [38]:
con.sql("""
    CREATE OR REPLACE TABLE temp_tbl AS
	WITH nbrs AS (
        SELECT PROV, geometry FROM hucs
        WHERE PROV IS NOT NULL
        ), 
	getprov AS (
        SELECT OBJECTID, WATERSHED_CODE, coalesce(hucs.PROV, nbrs.PROV) AS PROV, hucs.geometry
		FROM hucs
		JOIN nbrs
			ON ST_Intersects(hucs.geometry, nbrs.geometry)
		WHERE hucs.PROV IS NULL
        )
    SELECT OBJECTID, first(WATERSHED_CODE) AS WSC, histogram(PROV) AS PROV_COUNT
    FROM getprov
    GROUP BY OBJECTID
""")
con.sql("SELECT * FROM temp_tbl")

┌──────────┬─────────┬───────────────────────────────────┐
│ OBJECTID │   WSC   │            PROV_COUNT             │
│  int64   │ varchar │       map(varchar, ubigint)       │
├──────────┼─────────┼───────────────────────────────────┤
│      217 │ QC0218  │ {Gulf of St Lawrence=1, Québec=2} │
│      256 │ QC0257  │ {Gulf of St Lawrence=1, Québec=3} │
│      260 │ QC0261  │ {Gulf of St Lawrence=1, Québec=3} │
│      279 │ QC0280  │ {Gulf of St Lawrence=2, Québec=3} │
│     1072 │ NS01073 │ {Bay of Fundy=1, Nova Scotia=6}   │
│     1115 │ NS01116 │ {Bay of Fundy=3, Nova Scotia=2}   │
│     1154 │ NS01155 │ {Nova Scotia=4}                   │
│     1157 │ NS01158 │ {Nova Scotia=2}                   │
│     1179 │ NS01180 │ {Nova Scotia=4}                   │
│     1261 │ NS01262 │ {Nova Scotia=3}                   │
│       ·  │    ·    │        ·                          │
│       ·  │    ·    │        ·                          │
│       ·  │    ·    │        ·                         

In [39]:
con.sql("""
	SELECT OBJECTID, PROV_COUNT, e.unnest.key, e.unnest.value
	FROM temp_tbl
	CROSS JOIN UNNEST(map_entries(temp_tbl.PROV_COUNT)) AS e
	ORDER BY OBJECTID
""")

┌──────────┬───────────────────────────────────┬─────────────────────┬────────┐
│ OBJECTID │            PROV_COUNT             │         key         │ value  │
│  int64   │       map(varchar, ubigint)       │       varchar       │ uint64 │
├──────────┼───────────────────────────────────┼─────────────────────┼────────┤
│      217 │ {Gulf of St Lawrence=1, Québec=2} │ Gulf of St Lawrence │      1 │
│      217 │ {Gulf of St Lawrence=1, Québec=2} │ Québec              │      2 │
│      256 │ {Gulf of St Lawrence=1, Québec=3} │ Gulf of St Lawrence │      1 │
│      256 │ {Gulf of St Lawrence=1, Québec=3} │ Québec              │      3 │
│      260 │ {Gulf of St Lawrence=1, Québec=3} │ Gulf of St Lawrence │      1 │
│      260 │ {Gulf of St Lawrence=1, Québec=3} │ Québec              │      3 │
│      279 │ {Gulf of St Lawrence=2, Québec=3} │ Gulf of St Lawrence │      2 │
│      279 │ {Gulf of St Lawrence=2, Québec=3} │ Québec              │      3 │
│     1072 │ {Bay of Fundy=1, Nova Scoti

In [40]:
con.sql("""
	CREATE OR REPLACE TABLE temp_tbl AS
	WITH tbl AS (
		SELECT OBJECTID, PROV_COUNT, e.unnest.key, e.unnest.value
		FROM temp_tbl
		CROSS JOIN UNNEST(map_entries(temp_tbl.PROV_COUNT)) AS e
		ORDER BY OBJECTID
    )
    SELECT * FROM (
		SELECT tbl.*, ROW_NUMBER() OVER (
			PARTITION BY OBJECTID
			ORDER BY value DESC, key ASC
		) AS rn
	FROM tbl)
	WHERE rn=1
""")
con.sql("SELECT * FROM temp_tbl")

┌──────────┬───────────────────────────────────┬──────────────────────┬────────┬───────┐
│ OBJECTID │            PROV_COUNT             │         key          │ value  │  rn   │
│  int64   │       map(varchar, ubigint)       │       varchar        │ uint64 │ int64 │
├──────────┼───────────────────────────────────┼──────────────────────┼────────┼───────┤
│     2383 │ {Bay of Fundy=1}                  │ Bay of Fundy         │      1 │     1 │
│     2384 │ {Bay of Fundy=5}                  │ Bay of Fundy         │      5 │     1 │
│     1157 │ {Nova Scotia=2}                   │ Nova Scotia          │      2 │     1 │
│     2000 │ {Bay of Fundy=1, New Brunswick=6} │ New Brunswick        │      6 │     1 │
│     2017 │ {New Brunswick=2}                 │ New Brunswick        │      2 │     1 │
│      279 │ {Gulf of St Lawrence=2, Québec=3} │ Québec               │      3 │     1 │
│     1115 │ {Bay of Fundy=3, Nova Scotia=2}   │ Bay of Fundy         │      3 │     1 │
│      256 │ {Gulf of

In [41]:
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
	SELECT hucs.OBJECTID, hucs.WATERSHED_CODE, hucs.MERGE_SRC, coalesce(hucs.PROV,temp_tbl.key) AS PROV, geometry
    FROM hucs
	LEFT JOIN temp_tbl
        ON hucs.OBJECTID=temp_tbl.OBJECTID
""")

In [42]:
con.sql(""" 
	SELECT * EXCLUDE geometry, ST_Area(geometry)
	FROM hucs
    WHERE PROV IS NULL
""")

┌──────────┬────────────────┬────────────────────┬─────────┬───────────────────┐
│ OBJECTID │ WATERSHED_CODE │     MERGE_SRC      │  PROV   │ st_area(geometry) │
│  int64   │    varchar     │      varchar       │ varchar │      double       │
├──────────┼────────────────┼────────────────────┼─────────┼───────────────────┤
│     2367 │ NS02368        │ hucs\ncc_hu_smooth │ NULL    │ 5162321.742959529 │
└──────────┴────────────────┴────────────────────┴─────────┴───────────────────┘

In [43]:
## manually inspected, has wetland data, associate with New Brunswick
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
	SELECT hucs.OBJECTID, hucs.WATERSHED_CODE, hucs.MERGE_SRC, coalesce(hucs.PROV,'New Brunswick') AS PROV, geometry
    FROM hucs
""")

In [44]:
con.sql("SELECT distinct(PROV), count(*) FROM hucs GROUP BY PROV ORDER BY PROV")

┌──────────────────────┬──────────────┐
│         PROV         │ count_star() │
│       varchar        │    int64     │
├──────────────────────┼──────────────┤
│ Bay of Fundy         │           15 │
│ Gulf of St Lawrence  │            9 │
│ New Brunswick        │          415 │
│ Nova Scotia          │          318 │
│ Prince Edward Island │           74 │
│ Québec               │         1551 │
└──────────────────────┴──────────────┘

In [ ]:
## update watershed codes to match the new PROV values
con.sql("""
    CREATE OR REPLACE TABLE hucs AS    
    WITH code_map(code, name) AS (
        VALUES
            ('GS', 'Gulf of St Lawrence'),
            ('BF', 'Bay of Fundy'),
			('NB', 'New Brunswick'),
			('QC', 'Québec'),
			('NS', 'Nova Scotia'),
			('PE', 'Prince Edward Island'),
    )
    SELECT
        h.OBJECTID,
        replace(h.WATERSHED_CODE, h.WATERSHED_CODE[:2], cm.code) AS WATERSHED_CODE,
        h.MERGE_SRC,
        h.PROV,
        h.geometry
    FROM hucs AS h
    LEFT JOIN code_map AS cm
      ON h.PROV = cm.name
    ORDER BY OBJECTID
""")
con.sql("SELECT * FROM hucs")

┌──────────┬────────────────┬──────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [46]:
## manual correction after visual review
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
	SELECT
        h.OBJECTID,
        CASE WHEN h.WATERSHED_CODE='BF01116' THEN 'NS01116' ELSE h.WATERSHED_CODE END AS WATERSHED_CODE,
        h.MERGE_SRC,
        CASE WHEN h.WATERSHED_CODE='BF01116' THEN 'Nova Scotia' ELSE h.PROV END AS PROV,
        h.geometry
    FROM hucs AS h
""")
con.sql("SELECT * FROM hucs WHERE WATERSHED_CODE='BF01116'"), con.sql("SELECT * FROM hucs WHERE WATERSHED_CODE='NS01116'")

(┌──────────┬────────────────┬───────────┬─────────┬──────────┐
 │ OBJECTID │ WATERSHED_CODE │ MERGE_SRC │  PROV   │ geometry │
 │  int64   │    varchar     │  varchar  │ varchar │ geometry │
 ├──────────┴────────────────┴───────────┴─────────┴──────────┤
 │                           0 rows                           │
 └────────────────────────────────────────────────────────────┘,
 ┌──────────┬────────────────┬────────────────────┬─────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [47]:
con.sql("""
	CREATE OR REPLACE TABLE hucs AS
	SELECT
        h.OBJECTID,
        h.WATERSHED_CODE,
        h.MERGE_SRC,
        CASE WHEN h.PROV='NS' THEN 'Nova Scotia' ELSE h.PROV END AS PROV,
        h.geometry
    FROM hucs AS h
""")
con.sql("SELECT * FROM hucs WHERE WATERSHED_CODE='BF01116'"), con.sql("SELECT * FROM hucs WHERE WATERSHED_CODE='NS01116'")

(┌──────────┬────────────────┬───────────┬─────────┬──────────┐
 │ OBJECTID │ WATERSHED_CODE │ MERGE_SRC │  PROV   │ geometry │
 │  int64   │    varchar     │  varchar  │ varchar │ geometry │
 ├──────────┴────────────────┴───────────┴─────────┴──────────┤
 │                           0 rows                           │
 └────────────────────────────────────────────────────────────┘,
 ┌──────────┬────────────────┬────────────────────┬─────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [30]:
out_f = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_hucs.parquet"
# con.sql(f"COPY (SELECT * FROM hucs) TO '{out_f}' (FORMAT PARQUET)")
sql = f"""CREATE OR REPLACE TABLE hucs AS
SELECT * FROM read_parquet('{out_f}')
"""
con.execute(sql)
con.sql('DESCRIBE hucs')

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ MERGE_SRC      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PROV           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

#### Calculate per watershed wetland energy, protected area, unavailable, etc.

In [32]:
hucs = con.sql(f"SELECT {hucidfld} from hucs").df().values.tolist()
hucs = sorted([item for items in hucs for item in items])
len(hucs), hucs[:10]

(2382,
 ['BF02347',
  'BF02348',
  'BF02350',
  'BF02352',
  'BF02353',
  'BF02354',
  'BF02380',
  'BF02384',
  'BF02385',
  'BF02395'])

##### wetlands

In [49]:
con.execute("""
	CREATE OR REPLACE TABLE wetlands (
		{0} VARCHAR,
		{1} VARCHAR,
		geometry VARCHAR,
	)
""".format(wetattrfld, hucidfld))
con.sql('DESCRIBE wetlands')

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ CLASS_NAME     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [50]:
batch_size = 3
threads = []
print(len(hucs),hucs)
for i in range(len(hucs)):
    huc = hucs[i]
    threads.append(Thread(target = write_from_thread,
                            args = (con, nwiurl, 'wetlands',),
                            name = huc))
print(len(threads))

2382 ['BF02347', 'BF02348', 'BF02350', 'BF02352', 'BF02353', 'BF02354', 'BF02380', 'BF02384', 'BF02385', 'BF02395', 'BF02396', 'BF02397', 'BF02398', 'BF02399', 'GS02400', 'GS02401', 'GS02402', 'GS02403', 'GS02404', 'GS02405', 'GS02406', 'GS02407', 'GS02408', 'NB01117', 'NB01118', 'NB01119', 'NB01120', 'NB01121', 'NB01122', 'NB01123', 'NB01124', 'NB01125', 'NB01126', 'NB01127', 'NB01128', 'NB01129', 'NB01130', 'NB01131', 'NB01132', 'NB01133', 'NB01134', 'NB01135', 'NB01136', 'NB01137', 'NB01138', 'NB01139', 'NB01140', 'NB01141', 'NB01142', 'NB01143', 'NB01423', 'NB01424', 'NB01425', 'NB01426', 'NB01427', 'NB01428', 'NB01429', 'NB01430', 'NB01431', 'NB01432', 'NB01433', 'NB01434', 'NB01435', 'NB01436', 'NB01437', 'NB01438', 'NB01439', 'NB01440', 'NB01441', 'NB01442', 'NB01443', 'NB01444', 'NB01445', 'NB01446', 'NB01447', 'NB01448', 'NB01449', 'NB01450', 'NB01451', 'NB01452', 'NB01453', 'NB01455', 'NB01456', 'NB01457', 'NB01458', 'NB01459', 'NB01460', 'NB01461', 'NB01462', 'NB01463', 'NB0

In [ ]:
%%time
# Run threads in batches of 3
for i in range(0, len(threads), batch_size):
    batch = threads[i:i+batch_size]
    for thread in batch:
        thread.start()
    for thread in batch:
        thread.join()
    # Optional: short sleep between batches
    # sleep(0.1)

In [109]:
## Import wetland cross-walk data and assign classes to the nwi table
con.sql(f"""CREATE OR REPLACE TABLE crossnwi AS
        (UNPIVOT (FROM (SELECT * FROM read_json_auto('{crossWalk_json}', maximum_object_size=100000000))) ON COLUMNS(*))""")
con.sql("""CREATE OR REPLACE TABLE crossnwi AS
        SELECT name, UNNEST(value) AS value FROM crossnwi""")
# con.sql(f"""CREATE OR REPLACE TABLE wetlands AS
#         SELECT name, {wet_flds[-1]}, ST_GeomFromText(geometry) AS geometry FROM wetlands
#         LEFT JOIN crossnwi ON wetlands.{wetattrfld} LIKE crossnwi.value
#         """)
# con.sql(f"""
#         CREATE OR REPLACE TABLE wetlands AS
#         SELECT replace(wetlands.name, '_', '') AS name, {wet_flds[-1]}, ST_Area(geometry)*0.0001 AS ha, kcal, kcal*ha AS avalNrgy, st_buffer(geometry,0) AS geometry FROM wetlands
#         LEFT JOIN read_csv_auto('{nrgy_csv}') ON replace(wetlands.name, '_', '') = read_csv_auto.habitatType
#         WHERE wetlands.name IS NOT NULL
#         """)
print(con.sql('SELECT distinct(name) FROM wetlands'))

┌───────────────────────┐
│         name          │
│        varchar        │
├───────────────────────┤
│ SaltMarshNonDominant  │
│ FreshMarsh            │
│ FreshwaterWoody       │
│ DeepwaterFresh        │
│ FreshShallowOpenWater │
│ MudflatSalt           │
└───────────────────────┘



In [32]:
con.sql("""
    DELETE FROM wetlands
    WHERE ST_Extent(geometry) IS NULL
""")

In [ ]:
con.sql(f"""
        CREATE OR REPLACE TABLE wetlands AS
		SELECT wetlands.* EXCLUDE geometry, PROV, wetlands.geometry FROM wetlands
        JOIN hucs
		ON hucs.{hucidfld}=wetlands.{hucidfld}
""")
con.sql("""
		SELECT PROV, sum(kcal)
        FROM wetlands
		GROUP BY PROV
""")

┌──────────────────────┬────────────────┐
│         PROV         │   sum(kcal)    │
│       varchar        │     double     │
├──────────────────────┼────────────────┤
│ Gulf of St Lawrence  │   1960264268.0 │
│ New Brunswick        │  31439817859.0 │
│ Prince Edward Island │   2914757921.0 │
│ Bay of Fundy         │    703829676.0 │
│ Nova Scotia          │  20364128442.0 │
│ Québec               │ 160015154945.0 │
└──────────────────────┴────────────────┘

In [34]:
con.sql("""
    CREATE OR REPLACE TABLE wetlands AS
    WITH extnt AS (
        SELECT ST_Extent(ST_Extent_Agg(geometry))::BOX_2D AS box2d FROM wetlands
    )
    SELECT wetlands.* EXCLUDE geometry,
	ST_Hilbert(geometry, extnt.box2d) AS hilbert,
    geometry
    FROM wetlands, extnt
    ORDER BY hilbert
""")
con.sql("DESCRIBE wetlands")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ha             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ kcal           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ avalNrgy       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ PROV           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hilbert        │ UINTEGER    │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [33]:
out_f = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\wet_by_huc.parquet"
# sql = f"COPY (SELECT * FROM wetlands) TO '{out_f}' (FORMAT PARQUET)"
sql = f"""CREATE OR REPLACE TABLE wetlands AS
SELECT * FROM read_parquet('{out_f}')
"""
con.execute(sql)
con.sql("DESCRIBE wetlands")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ha             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ kcal           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ avalNrgy       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ PROV           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hilbert        │ UINTEGER    │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

##### protected area

In [34]:
cols = con.sql(f"DESCRIBE (SELECT * FROM read_parquet('{protLands}'))").df()[['column_name','column_type']].values
# cols = con.sql(f"DESCRIBE (SELECT * FROM read_parquet('{protLands}'))").df().column_name.to_list()
cols

array([['OBJECTID', 'BIGINT'],
       ['PARENT_ID', 'INTEGER'],
       ['NAME_E', 'VARCHAR'],
       ['NAME_F', 'VARCHAR'],
       ['NAME_IND', 'VARCHAR'],
       ['ZONE_ID', 'INTEGER'],
       ['ZONEDESC_E', 'VARCHAR'],
       ['BIOME', 'VARCHAR'],
       ['JUR_ID', 'VARCHAR'],
       ['PA_OECM_DF', 'INTEGER'],
       ['IUCN_CAT', 'INTEGER'],
       ['IPCA', 'INTEGER'],
       ['IND_TERM', 'VARCHAR'],
       ['O_AREA_HA', 'DOUBLE'],
       ['LOC', 'INTEGER'],
       ['MECH_E', 'VARCHAR'],
       ['TYPE_E', 'VARCHAR'],
       ['OWNER_TYPE', 'INTEGER'],
       ['OWNER_E', 'VARCHAR'],
       ['GOV_TYPE', 'INTEGER'],
       ['MGMT_E', 'VARCHAR'],
       ['STATUS', 'INTEGER'],
       ['ESTYEAR', 'INTEGER'],
       ['QUALYEAR', 'INTEGER'],
       ['DELISTYEAR', 'INTEGER'],
       ['SUBSRIGHT', 'VARCHAR'],
       ['CONS_OBJ', 'INTEGER'],
       ['NO_TAKE', 'INTEGER'],
       ['NO_TAKE_HA', 'DOUBLE'],
       ['MPLAN', 'INTEGER'],
       ['MPLAN_REF', 'VARCHAR'],
       ['M_EFF', 'INTEGER'],
 

In [35]:
cols = ', '.join([f"{c} {d}" for c,d in cols[:-1]]) # map data types or change sql_picker function to specify fields to read in from parquet
print(cols.replace(', ','\n'))

OBJECTID BIGINT
PARENT_ID INTEGER
NAME_E VARCHAR
NAME_F VARCHAR
NAME_IND VARCHAR
ZONE_ID INTEGER
ZONEDESC_E VARCHAR
BIOME VARCHAR
JUR_ID VARCHAR
PA_OECM_DF INTEGER
IUCN_CAT INTEGER
IPCA INTEGER
IND_TERM VARCHAR
O_AREA_HA DOUBLE
LOC INTEGER
MECH_E VARCHAR
TYPE_E VARCHAR
OWNER_TYPE INTEGER
OWNER_E VARCHAR
GOV_TYPE INTEGER
MGMT_E VARCHAR
STATUS INTEGER
ESTYEAR INTEGER
QUALYEAR INTEGER
DELISTYEAR INTEGER
SUBSRIGHT VARCHAR
CONS_OBJ INTEGER
NO_TAKE INTEGER
NO_TAKE_HA DOUBLE
MPLAN INTEGER
MPLAN_REF VARCHAR
M_EFF INTEGER
AUDITYEAR INTEGER
AUDITRES VARCHAR
PROVIDER INTEGER


In [36]:
sql = f"""
	CREATE OR REPLACE TABLE protected (
	{hucidfld} VARCHAR,
    {cols},
	geometry VARCHAR,
)
"""
con.sql(sql)

In [37]:
batch_size = 20
threads=[]
for i in range(len(hucs)):
        huc = hucs[i]
        threads.append(Thread(target = write_from_thread,
                                args = (con, protLands, 'protected',),
                                name = huc))
print(len(hucs), len(threads))

2382 2382


In [38]:
%%time
if len(threads)>0:
    for i in range(0, len(threads), batch_size):
        batch = threads[i:i+batch_size]
        for thread in batch:
            thread.start()
        for thread in batch:
            thread.join()

CPU times: total: 1h 19min 7s
Wall time: 4min 59s


In [39]:
con.sql("""CREATE OR REPLACE TABLE protected AS
        SELECT * EXCLUDE geometry, ST_GeomFromText(geometry) AS geometry FROM protected""")
con.sql(f"SELECT * FROM protected")

┌────────────────┬──────────┬───────────┬──────────────────────┬──────────────────────┬──────────┬───────────┬──────────────────┬─────────┬─────────────────┬────────────┬──────────┬───────┬──────────┬────────────────────┬───────┬──────────────────────┬──────────────────────┬────────────┬──────────────────────┬──────────┬──────────────────────┬────────┬─────────┬──────────┬────────────┬───────────┬──────────┬─────────┬────────────┬───────┬──────────────────────┬───────┬───────────┬──────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [56]:
con.sql("""
CREATE OR REPLACE TABLE protected AS
SELECT * EXCLUDE geometry, ST_Area(geometry)*0.0001 AS protected_ha, geometry FROM protected
""")

In [57]:
con.sql("""
    CREATE OR REPLACE TABLE protected AS
    WITH extnt AS (
        SELECT ST_Extent(ST_Extent_Agg(geometry))::BOX_2D AS box2d FROM protected
    )
    SELECT WATERSHED_CODE, OBJECTID, NAME_E, LOC, protected_ha,	
	ST_Hilbert(geometry, extnt.box2d) AS hilbert,
    geometry
    FROM protected, extnt
    ORDER BY hilbert
""")
con.sql("""DESCRIBE protected""")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OBJECTID       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ NAME_E         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LOC            │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ protected_ha   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ hilbert        │ UINTEGER    │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [58]:
out_f = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\prot_by_huc.parquet"
sql = f"COPY (SELECT * FROM protected) TO '{out_f}' (FORMAT PARQUET)"
# sql = f"""CREATE OR REPLACE TABLE protected AS
# SELECT * FROM read_parquet('{out_f}')
# """
con.execute(sql)
con.sql("""DESCRIBE protected""")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OBJECTID       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ NAME_E         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LOC            │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ protected_ha   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ hilbert        │ UINTEGER    │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [42]:
con.sql("""
        SELECT * EXCLUDE geometry
		FROM protected
        WHERE LOC <= 13
""")

┌────────────────┬──────────┬───────────────────────────────────────────────────────────────────┬───────┬────────────┐
│ WATERSHED_CODE │ OBJECTID │                              NAME_E                               │  LOC  │  hilbert   │
│    varchar     │  int64   │                              varchar                              │ int32 │   uint32   │
├────────────────┼──────────┼───────────────────────────────────────────────────────────────────┼───────┼────────────┤
│ QC026          │     9081 │ Mattawa White-Tailed Deer Yard                                    │    11 │   73996689 │
│ QC027          │     9081 │ Mattawa White-Tailed Deer Yard                                    │    11 │   74191809 │
│ QC027          │    12902 │ 08151R007 Biological Refuge                                       │    11 │   76635263 │
│ QC026          │    13791 │ Land set aside Basses-Collines-du-Ruisseau-Serpent                │    11 │   76999179 │
│ QC026          │    10164 │ Proposed Basses-Co

In [45]:
(
    con.sql("""
        SELECT count(*)
		FROM protected
        WHERE LOC <= 13
	"""),
    con.sql("""
        SELECT count(*)
		FROM protected
        WHERE LOC > 13
	""")
)

(┌──────────────┐
 │ count_star() │
 │    int64     │
 ├──────────────┤
 │        10218 │
 └──────────────┘,
 ┌──────────────┐
 │ count_star() │
 │    int64     │
 ├──────────────┤
 │          158 │
 └──────────────┘)

In [46]:
con.sql("""
        SELECT count(distinct(WATERSHED_CODE))
		FROM protected
        WHERE LOC <= 13
""")

┌────────────────────────────────┐
│ count(DISTINCT WATERSHED_CODE) │
│             int64              │
├────────────────────────────────┤
│                           1886 │
└────────────────────────────────┘

In [ ]:
#### EXCLUDE MARINE PROTECTED AREAS (locations > 13)
codes = [r[0] for r in con.sql(
    f"SELECT DISTINCT {hucidfld} FROM protected WHERE LOC <= 13"
).fetchall()]
len(codes)

1886

In [ ]:
con.sql("DROP TABLE IF EXISTS protwetlands")

for i, code in enumerate(codes, start=1):
    print(f"\r{i}/{len(codes)}: {code}", end=' '*len(code))
    sql = f"""
        SELECT name, wetlands.{hucidfld}, kcal,
               ST_Intersection(p.geometry, wetlands.geometry) AS geometry
        FROM (
            SELECT ST_Union_Agg(geometry) AS geometry
            FROM protected
            WHERE LOC <= 13 AND {hucidfld} = ?
        ) AS p
        JOIN wetlands ON ST_Intersects(wetlands.geometry, p.geometry)
    """
    if i == 1:
        con.sql(f"CREATE TABLE protwetlands AS {sql}", params=[code])
    else:
        con.sql(f"INSERT INTO protwetlands {sql}", params=[code])

con.sql("SELECT count(name) FROM protwetlands")

1885/1886: NS01346       

┌───────────────┐
│ count("name") │
│     int64     │
├───────────────┤
│        217475 │
└───────────────┘

In [49]:
con.sql("""
CREATE OR REPLACE TABLE protwetlands AS
SELECT DISTINCT geometry, name, {0}, ST_Area(geometry)*0.0001 AS ProtHabHa, kcal, kcal*ProtHabHa AS protNrgy FROM protwetlands
""".format(hucidfld))

In [50]:
i = con.sql("SELECT sum(protNrgy) FROM protwetlands").fetchone()[0]
k = con.sql("SELECT sum(kcal) FROM wetlands").fetchone()[0]
print(f"{i:,.2f} protected wetland kcal\n{k:,.2f} total wetland kcal\n= {round((i/k)*100,2)}%")

56,408,437,171.78 protected wetland kcal
217,397,953,111.00 total wetland kcal
= 25.95%


In [51]:
out_f = r"D:\ABDUBounds\model_output\protwetlands.parquet"
con.sql(f"COPY (SELECT * FROM protwetlands) TO '{out_f}' (FORMAT PARQUET)")
# sql = f"""CREATE OR REPLACE TABLE protwetlands AS
# SELECT * FROM read_parquet('{out_f}')
# """
# con.execute(sql)
con.sql("DESCRIBE protwetlands")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
│ name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ProtHabHa      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ kcal           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ protNrgy       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

##### urban/roads land

In [52]:
cols = con.sql(f"DESCRIBE (SELECT * EXCLUDE geometry FROM read_parquet('{urbanMask}'))").df()[['column_name','column_type']].values
# cols = con.sql(f"DESCRIBE (SELECT * FROM read_parquet('{protLands}'))").df().column_name.to_list()
cols = ', '.join([f"{c} {d}" for c,d in cols]) # map data types or change sql_picker function to specify fields to read in from parquet
print(cols.replace(', ','\n'))

OBJECTID BIGINT
Id INTEGER
gridcode INTEGER
Shape_Length DOUBLE
Shape_Area DOUBLE
geometry_bbox STRUCT(xmax DOUBLE
xmin DOUBLE
ymax DOUBLE
ymin DOUBLE)


In [ ]:
sql = f"""
	CREATE OR REPLACE TABLE urban (
		{hucidfld} VARCHAR,
		{cols},
		geometry VARCHAR,
	)
"""
con.sql(sql)

In [54]:
batch_size = 10
threads = []
for i in range(len(hucs)):
	huc = hucs[i]
	threads.append(Thread(target = write_from_thread,
							args = (con, urbanMask, 'urban',),
							name = huc))
print(len(threads))

2382


In [55]:
%%time
if len(threads)>0:
    for i in range(0, len(threads), batch_size):
        batch = threads[i:i+batch_size]
        for thread in batch:
            thread.start()
        for thread in batch:
            thread.join()

CPU times: total: 1d 19h 59min 16s
Wall time: 3h 39min 55s


In [59]:
con.sql("""
CREATE OR REPLACE TABLE urban AS
SELECT * EXCLUDE geometry, {0} FROM urban
""".format('ST_GeomFromText(geometry) AS geometry' if len(threads)>0 else 'geometry'))

con.sql("""
CREATE OR REPLACE TABLE urban AS
SELECT * EXCLUDE geometry, ST_Area(geometry)*0.0001 AS urbanHa, geometry FROM urban
""")

In [60]:
con.sql("""
    CREATE OR REPLACE TABLE urban AS
    WITH extnt AS (
        SELECT ST_Extent(ST_Extent_Agg(geometry))::BOX_2D AS box2d FROM urban
    )
    SELECT urban.* EXCLUDE geometry,
	ST_Hilbert(geometry, extnt.box2d) AS hilbert,
    geometry
    FROM urban, extnt
    ORDER BY hilbert
""")
con.sql("DESCRIBE urban")

┌────────────────┬────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │                        column_type                         │  null   │   key   │ default │  extra  │
│    varchar     │                          varchar                           │ varchar │ varchar │ varchar │ varchar │
├────────────────┼────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ WATERSHED_CODE │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ OBJECTID       │ BIGINT                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ Id             │ INTEGER                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ gridcode       │ INTEGER                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ Shape_Length   │ DOUBLE               

In [61]:
out_f = r"D:\ABDUBounds\model_output\urban.parquet"
con.sql(f"COPY (SELECT * FROM urban) TO '{out_f}' (FORMAT PARQUET)")
# sql = f"""CREATE OR REPLACE TABLE urban AS
# SELECT * FROM read_parquet('{out_f}')
# """
# con.execute(sql)
# con.sql("DESCRIBE urban")

In [62]:
con.sql("""
CREATE OR REPLACE TABLE unavailable AS
SELECT {0}, ST_Area(geometry)*0.0001 AS unavailHa, ST_Union_Agg(geometry) as geometry FROM
(
SELECT {0}, geometry FROM urban
UNION ALL
SELECT {0}, geometry from protected
)
group by {0}, geometry
""".format(hucidfld))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE urbanwetlands AS
SELECT name, wetlands.{0}, kcal, ST_Intersection(urban.geometry, wetlands.geometry) as geometry
FROM urban
JOIN wetlands ON ST_Intersects(wetlands.geometry, urban.geometry)
""".format(hucidfld))

In [63]:
con.sql("SELECT count(*) FROM urban WHERE ST_Area(geometry)<=0.0")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

In [64]:
codes = [r[0] for r in con.sql(
    f"SELECT DISTINCT {hucidfld} FROM urban"
).fetchall()]
len(codes)

2137

In [67]:
con.sql("DROP TABLE IF EXISTS urbanwetlands")

for i, code in enumerate(codes, start=1):
    print(f"\r{i}/{len(codes)}: {code}", end=' '*len(code))
    sql = f"""
        SELECT name, wetlands.{hucidfld}, kcal,
               ST_Intersection(p.geometry, wetlands.geometry) AS geometry
        FROM (
            SELECT ST_Union_Agg(geometry) AS geometry
            FROM urban
            WHERE {hucidfld} = ?
        ) AS p
        JOIN wetlands ON ST_Intersects(wetlands.geometry, p.geometry)
    """
    if i == 1:
        con.sql(f"CREATE TABLE urbanwetlands AS {sql}", params=[code])
    else:
        con.sql(f"INSERT INTO urbanwetlands {sql}", params=[code])

con.sql("SELECT count(name) FROM urbanwetlands")

2137/2137: NS01307       

┌───────────────┐
│ count("name") │
│     int64     │
├───────────────┤
│        120479 │
└───────────────┘

In [71]:
con.sql("SELECT count(*) FROM urbanwetlands WHERE ST_Area(geometry)<=0.0")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2045 │
└──────────────┘

In [73]:
con.sql("""
    DELETE FROM urbanwetlands
    WHERE ST_Area(geometry)<=0.0
""")

In [74]:
con.sql("""
    SELECT count(*) FROM urbanwetlands
    WHERE ST_Extent(geometry) IS NULL
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

In [75]:
con.sql("""
CREATE OR REPLACE TABLE urbanwetlands AS
SELECT DISTINCT geometry, name, {0}, ST_Area(geometry)*0.0001 AS UrbanHabHa, kcal, kcal*UrbanHabHa AS urbanNrgy FROM urbanwetlands
""".format(hucidfld))

In [76]:
i = con.sql("SELECT sum(urbanNrgy) FROM urbanwetlands").fetchone()[0]
print(f"{i:,.2f} urban wetland kcal\n{k:,.2f} total wetland kcal\n= {round((i/k)*100,2)}%")

3,893,755,169.66 urban wetland kcal
217,397,953,111.00 total wetland kcal
= 1.79%


In [77]:
con.sql("""
    CREATE OR REPLACE TABLE urbanwetlands AS
    WITH extnt AS (
        SELECT ST_Extent(ST_Extent_Agg(geometry))::BOX_2D AS box2d FROM urbanwetlands
    )
    SELECT urbanwetlands.* EXCLUDE geometry,
	ST_Hilbert(geometry, extnt.box2d) AS hilbert,
    geometry
    FROM urbanwetlands, extnt
    ORDER BY hilbert
""")
con.sql("DESCRIBE urbanwetlands")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ UrbanHabHa     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ kcal           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ urbanNrgy      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ hilbert        │ UINTEGER    │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [78]:
out_f = r"D:\ABDUBounds\model_output\urbanwetlands.parquet"
con.sql(f"COPY (SELECT * FROM urbanwetlands) TO '{out_f}' (FORMAT PARQUET)")
# sql = f"""CREATE OR REPLACE TABLE urbanwetlands AS
# SELECT * FROM read_parquet('{out_f}')
# """
# con.execute(sql)
# con.sql("DESCRIBE urbanwetlands")

#### Demand calculation

In [ ]:
'''
Aggregate demand by province.
Sum energy of wetlands in watersheds per province.
Calculate percent of each watershed's energy of total in province.
Take that percent of total demand of province.
'''

In [79]:
con.sql("DESCRIBE demand")

┌─────────────┬────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │                        column_type                         │  null   │   key   │ default │  extra  │
│   varchar   │                          varchar                           │ varchar │ varchar │ varchar │ varchar │
├─────────────┼────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID    │ BIGINT                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ species     │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ fips        │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ CODE        │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ LTADUD      │ INTEGER                                         

In [80]:
con.sql("""
	SELECT Name_EN, sum(LTADemand), sum(X80Demand), sum(LTADUD)
	FROM demand
	WHERE species = 'All'
	GROUP BY Name_EN
""")

┌─────────────────────────┬────────────────────┬────────────────────┬─────────────┐
│         Name_EN         │   sum(LTADemand)   │   sum(X80Demand)   │ sum(LTADUD) │
│         varchar         │       double       │       double       │   int128    │
├─────────────────────────┼────────────────────┼────────────────────┼─────────────┤
│ Newfoundland & Labrador │  5018223498.432012 │   5437173186.32013 │    15885053 │
│ Yukon                   │     49725181.49929 │     59848484.49494 │      151437 │
│ Nova Scotia             │ 11227568927.549702 │ 12605459612.932201 │    37168804 │
│ British Columbia        │   4138707420.26285 │ 5239796273.4669695 │    14542959 │
│ Québec                  │  37245826055.02518 │   43798348861.9348 │   123750598 │
│ Saskatchewan            │  49286176741.12599 │  63452136146.76441 │   163446092 │
│ Northwest Territories   │     65878412.61239 │     78689778.56286 │      231193 │
│ Prince Edward Island    │     2560522980.859 │ 3007896094.1729994 │     84

In [81]:
con.sql(""" 
	WITH d AS (
        SELECT * FROM demand
        JOIN hucs
        ON hucs.PROV=demand.Name_EN)
    SELECT PROV, code, count(*)
	FROM d
	GROUP BY PROV, code
""")

┌──────────────────────┬─────────┬──────────────┐
│         PROV         │  CODE   │ count_star() │
│       varchar        │ varchar │    int64     │
├──────────────────────┼─────────┼──────────────┤
│ Québec               │ 4B      │       440484 │
│ Nova Scotia          │ 4B      │        31900 │
│ Prince Edward Island │ 4B      │          888 │
│ New Brunswick        │ 4B      │        19920 │
└──────────────────────┴─────────┴──────────────┘

In [83]:
con.sql("SELECT distinct(PROV) from hucs ORDER BY PROV"), con.sql("SELECT distinct(PROV) from wetlands ORDER BY PROV")

(┌──────────────────────┐
 │         PROV         │
 │       varchar        │
 ├──────────────────────┤
 │ Bay of Fundy         │
 │ Gulf of St Lawrence  │
 │ New Brunswick        │
 │ Nova Scotia          │
 │ Prince Edward Island │
 │ Québec               │
 └──────────────────────┘,
 ┌──────────────────────┐
 │         PROV         │
 │       varchar        │
 ├──────────────────────┤
 │ Bay of Fundy         │
 │ Gulf of St Lawrence  │
 │ New Brunswick        │
 │ Nova Scotia          │
 │ Prince Edward Island │
 │ Québec               │
 └──────────────────────┘)

In [84]:
con.sql("DESCRIBE wetlands").df().column_name.to_list()

['name',
 'WATERSHED_CODE',
 'ha',
 'kcal',
 'avalNrgy',
 'PROV',
 'hilbert',
 'geometry']

In [85]:
con.sql("""
	CREATE OR REPLACE TABLE cn_prov AS
    WITH e AS (
        SELECT PROV, sum(avalNrgy) AS total_kcal
		FROM wetlands
		GROUP BY PROV
	), d AS (
		SELECT Name_EN, species,
        sum(LTADUD) AS LTADUD, sum(X80DUD) AS X80DUD, sum(LTAPopObj) AS LTAPopObj, sum(X80PopObj) AS X80PopObj, sum(LTADemand) AS LTADemand, sum(X80Demand) AS X80Demand
		FROM demand
		GROUP BY Name_EN, species	
	)
    SELECT * FROM e
	LEFT JOIN d
    ON e.PROV=d.Name_EN
""")
con.sql("SELECT * FROM cn_prov ORDER BY PROV, species")

┌─────────────────────┬────────────────────┬───────────────┬─────────┬───────────┬───────────┬───────────┬───────────┬────────────────────┬────────────────────┐
│        PROV         │     total_kcal     │    Name_EN    │ species │  LTADUD   │  X80DUD   │ LTAPopObj │ X80PopObj │     LTADemand      │     X80Demand      │
│       varchar       │       double       │    varchar    │ varchar │  int128   │  int128   │  int128   │  int128   │       double       │       double       │
├─────────────────────┼────────────────────┼───────────────┼─────────┼───────────┼───────────┼───────────┼───────────┼────────────────────┼────────────────────┤
│ Bay of Fundy        │ 3191790484.7305875 │ NULL          │ NULL    │      NULL │      NULL │      NULL │      NULL │               NULL │               NULL │
│ Gulf of St Lawrence │  8843321312.166683 │ NULL          │ NULL    │      NULL │      NULL │      NULL │      NULL │               NULL │               NULL │
│ New Brunswick       │  691038825

In [86]:
con.sql(f"""
	CREATE OR REPLACE TABLE rdydemand AS 
	SELECT name, {hucidfld}, wetlands.PROV, species, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, kcal, avalNrgy, (wetlands.avalNrgy/total_kcal) as pct
	FROM wetlands
    LEFT JOIN cn_prov
    ON wetlands.PROV=cn_prov.PROV
""")
con.sql("SELECT * FROM rdydemand ORDER BY PROV, species")

┌───────────────────────┬────────────────┬─────────────────────┬─────────┬────────┬───────────┬───────────┬────────┬───────────┬───────────┬──────────┬─────────────────────┬────────────────────────┐
│         name          │ WATERSHED_CODE │        PROV         │ species │ LTADUD │ LTADemand │ LTAPopObj │ X80DUD │ X80Demand │ X80PopObj │   kcal   │      avalNrgy       │          pct           │
│        varchar        │    varchar     │       varchar       │ varchar │ int128 │  double   │  int128   │ int128 │  double   │  int128   │  double  │       double        │         double         │
├───────────────────────┼────────────────┼─────────────────────┼─────────┼────────┼───────────┼───────────┼────────┼───────────┼───────────┼──────────┼─────────────────────┼────────────────────────┤
│ DeepwaterFresh        │ BF02399        │ Bay of Fundy        │ NULL    │   NULL │      NULL │      NULL │   NULL │      NULL │      NULL │  61629.0 │   30.71343712807706 │  9.622635719671782e-09 │
│ Mud

In [87]:
con.sql(f"""CREATE OR REPLACE TABLE hucdemand AS (SELECT {hucidfld}, species,
    sum(pct * LTADUD) AS LTADUD,
    sum(pct * LTADemand) AS LTADemand,
    sum(pct * LTAPopObj) AS LTAPopObj,
    sum(pct * x80DUD) AS x80DUD,
    sum(pct * X80Demand) AS X80Demand,
    sum(pct * X80PopObj) AS X80PopObj,
    FROM rdydemand
    GROUP BY {hucidfld}, species)
""")
con.sql("SELECT * FROM hucdemand")

┌────────────────┬─────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┐
│ WATERSHED_CODE │ species │       LTADUD       │     LTADemand      │     LTAPopObj      │       x80DUD       │     X80Demand      │     X80PopObj      │
│    varchar     │ varchar │       double       │       double       │       double       │       double       │       double       │       double       │
├────────────────┼─────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┤
│ NB01794        │ AGWT    │  419.2372835199926 │ 54919.984731505974 │ 3.9203046811388367 │  535.7020384680461 │  70177.01147383734 │  5.009369945338947 │
│ NB01796        │ AGWT    │  315.6090137152436 │  41344.70595943604 │ 2.9512725669077486 │  403.2856777111589 │ 52830.457231232685 │ 3.7711395668553753 │
│ NB01756        │ WODU    │  2194.867956375564 │  487261.7957820587 │

In [88]:
con.sql('''CREATE OR REPLACE TABLE spp_demand AS
(SELECT * FROM
(PIVOT hucdemand
    on species
    USING {0}))
'''.format(', '.join([f"sum({i})" for i in con.sql("DESCRIBE hucdemand").df().column_name.to_list()[2:]])))

In [89]:
for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:]:
    con.sql(f"""ALTER TABLE spp_demand RENAME COLUMN '{i}' TO '{i.replace('sum(','').replace(')','')}'""")
con.sql("SELECT * FROM spp_demand")

┌────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬───────

In [90]:
spp_list

['ABDU', 'AGWT', 'AMWI', 'All', 'BWTE', 'GADW', 'MALL', 'NOPI', 'NSHO', 'WODU']

In [91]:
con.sql("SELECT distinct(species) FROM hucdemand ORDER by species")

┌─────────┐
│ species │
│ varchar │
├─────────┤
│ ABDU    │
│ AGWT    │
│ AMWI    │
│ All     │
│ BWTE    │
│ GADW    │
│ MALL    │
│ NOPI    │
│ NSHO    │
│ WODU    │
│ NULL    │
├─────────┤
│ 11 rows │
└─────────┘

# Summarize at huc level

In [92]:
con.execute(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT hucs.{hucidfld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, hucs.geometry
FROM hucs
LEFT JOIN (SELECT * FROM hucdemand WHERE species = 'All') as d
ON hucs.{hucidfld} = d.{hucidfld}
ORDER by hucs.{hucidfld}
""")

In [93]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
cols

'LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj'

In [94]:
# Specified selection in a cell or two below.  Many don't need geometry at this later point.  Joining is by hucs so selecting
# only the required columns makes the join go much faster.
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
sum(avalNrgy) as tothabitat_kcal,
athuclevel.geometry
FROM athuclevel
LEFT JOIN wetlands on athuclevel.{hucidfld}=wetlands.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {cols}, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [95]:
con.sql("select count(*) from athuclevel"), con.sql("select count(*) from hucs"), con.sql(f"select count(distinct({hucidfld})) from hucdemand WHERE species = 'All'")

(┌──────────────┐
 │ count_star() │
 │    int64     │
 ├──────────────┤
 │         2382 │
 └──────────────┘,
 ┌──────────────┐
 │ count_star() │
 │    int64     │
 ├──────────────┤
 │         2382 │
 └──────────────┘,
 ┌────────────────────────────────┐
 │ count(DISTINCT WATERSHED_CODE) │
 │             int64              │
 ├────────────────────────────────┤
 │                           2359 │
 └────────────────────────────────┘)

In [96]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
sum(protected_ha) as protected_ha,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (
	SELECT {hucidfld}, sum(protected_ha) AS protected_ha
	FROM protected
    WHERE LOC <= 13
	GROUP BY {hucidfld}
) as p
ON athuclevel.{hucidfld} = p.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {cols}, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [97]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
sum(ProtHabHa) as protectedhabitat_ha,
sum(protNrgy) as protected_kcal,
athuclevel.geometry
FROM athuclevel
LEFT JOIN protwetlands on protwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {cols}, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [98]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
sum(urbanHa) as urbanHa,
athuclevel.geometry
FROM athuclevel
LEFT JOIN urban on urban.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {cols}, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [99]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
sum(urbanNrgy) as urbanNrgy,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (SELECT {hucidfld}, urbanNrgy FROM urbanwetlands) as urbanwetlands on urbanwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld},{cols}, athuclevel.geometry
""")

In [100]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
sum(unavailHa) as unavailHa,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (SELECT {hucidfld}, unavailHa FROM unavailable) as unavailable on unavailable.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {cols}, athuclevel.geometry
""")

In [101]:
cols = con.sql("DESCRIBE athuclevel").df().column_name.to_list()
cols = ', '.join(cols[1:-1])
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {cols},
{','.join([i for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:] if not 'All' in i])},
athuclevel.geometry
FROM athuclevel
LEFT JOIN spp_demand on spp_demand.{hucidfld} = athuclevel.{hucidfld}
ORDER by athuclevel.{hucidfld}
""")

In [102]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld},
ST_Area(geometry)*0.0001 wshed_ha, 
COALESCE(LTADUD, 0) dud_lta,
COALESCE(LTADemand,0) demand_lta_kcal, 
COALESCE(LTAPopObj,0) popobj_lta, 
COALESCE(X80DUD,0) dud_80th, 
COALESCE(X80Demand,0) demand_80th_kcal, 
COALESCE(X80PopObj,0) popobj_80th,
{','.join([f'COALESCE({i}, 0) {i.lower()}' for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:] if not 'All' in i])},
COALESCE(protected_ha,0) protected_ha,
COALESCE(tothabitat_kcal,0) tothabitat_kcal,
COALESCE(protected_kcal,0) protected_kcal,
COALESCE(protectedhabitat_ha,0) protectedhabitat_ha,
COALESCE(urbanHa,0) urbanHa, 
COALESCE(sum(urbanNrgy),0) urbanNrgy,
COALESCE(sum(unavailHa),0) unavailha,
COALESCE(tothabitat_kcal - demand_lta_kcal,0) surpdef_lta_kcal,
COALESCE(tothabitat_kcal - demand_80th_kcal,0) surpdef_80th_kcal,
athuclevel.geometry
FROM athuclevel
GROUP BY *
ORDER BY athuclevel.{hucidfld}
""")

In [103]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS 
SELECT *,
CASE WHEN 
demand_lta_kcal - protected_kcal > 0
THEN
demand_lta_kcal - protected_kcal
ELSE 0
END
AS nrgprot_lta_kcal,
CASE WHEN
demand_80th_kcal - protected_kcal > 0 
THEN
demand_80th_kcal - protected_kcal
ELSE 0
END
AS nrgprot_80th_kcal
FROM athuclevel
''')

In [88]:
45651056-87089480

-41438424

# Calculate weighted mean

In [104]:
'''
Calculate mean energy per ha weighted by wetland type
'''
con.sql(f'''
        CREATE OR REPLACE TABLE wtmean AS 
        SELECT huctotal.{hucidfld}, name, tothab_ha, (habname_ha/tothab_ha)*100 as pct_ha, avalNrgname/avalNrgtot as pct, hucnametotal.kcal * pct as wtmean FROM
        ((SELECT {hucidfld}, sum(ha) as tothab_ha, sum(avalNrgy) as avalNrgtot from wetlands group by {hucidfld}) huctotal
        join
        (SELECT {hucidfld}, name, kcal, sum(ha) as habname_ha, sum(avalNrgy) as avalNrgname from wetlands group by {hucidfld}, name, kcal) hucnametotal
        on hucnametotal.{hucidfld} = huctotal.{hucidfld})
''')
con.sql(f'''CREATE OR REPLACE TABLE wtmeanpivot AS
(select {hucidfld}, {','.join([i[0] for i in con.sql("SELECT distinct(name) FROM wtmean").fetchall()])} FROM
(pivot wtmean
    on name
    USING sum(wtmean)))
''')

In [105]:
con.sql(f'''CREATE OR REPLACE TABLE wtmeanbyhuc AS 
        SELECT {hucidfld},
        sum({' + '.join([f"COALESCE({i},0)" for i in con.sql("DESCRIBE wtmeanpivot").df().column_name.to_list()[1:]])}) as wtMean_kcal_per_ha
        FROM wtmeanpivot
        GROUP BY {hucidfld}
        ORDER BY {hucidfld}
''')
# con.sql(f'''create or replace table wtmeanbyhuc as
#         select {hucidfld}, DeepwaterFresh, FreshMarsh, FreshShallowOpenWater, FreshwaterWoody, ManagedFreshMarsh, ManagedFreshShallowOpenWater, ManagedFreshwaterAquaticBed,
#         sum(DeepwaterFresh + FreshMarsh + FreshShallowOpenWater + FreshwaterWoody + ManagedFreshMarsh +ManagedFreshShallowOpenWater + ManagedFreshwaterAquaticBed)
#         as wtmean from wtmeanpivot
#         group by {hucidfld}''')
con.sql(f"""SELECT * FROM wtmeanbyhuc""")

┌────────────────┬────────────────────┐
│ WATERSHED_CODE │ wtMean_kcal_per_ha │
│    varchar     │       double       │
├────────────────┼────────────────────┤
│ BF02347        │  75945.85921133739 │
│ BF02348        │  78680.96011774412 │
│ BF02350        │ 135608.33910284817 │
│ BF02352        │ 259012.97566390896 │
│ BF02353        │ 104377.96003644388 │
│ BF02354        │  92564.96372261312 │
│ BF02380        │  182898.7489410416 │
│ BF02384        │ 251024.83211322475 │
│ BF02385        │ 249305.26511990803 │
│ BF02395        │   78705.4841956499 │
│   ·            │           ·        │
│   ·            │           ·        │
│   ·            │           ·        │
│ QC0990         │ 196054.74713824556 │
│ QC0991         │  150291.7016318599 │
│ QC0992         │  129523.2959725668 │
│ QC0993         │  116766.8505732434 │
│ QC0994         │ 123003.56152826948 │
│ QC0995         │ 131329.13507765398 │
│ QC0996         │ 134102.97179419626 │
│ QC0997         │ 123634.75744476184 │


In [106]:
con.sql(f'''CREATE OR REPLACE TABLE habpivot AS
(select {hucidfld}, tothab_ha, {','.join([i[0]+'_pct_ha' for i in con.sql("SELECT distinct(name) FROM wtmean").fetchall()])} FROM
(pivot wtmean
    on name
    USING sum(pct_ha) AS pct_ha))
''')

In [107]:
con.sql(f"""SELECT * FROM habpivot ORDER BY {hucidfld}""")

┌────────────────┬────────────────────┬───────────────────────┬─────────────────────────────┬────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────┐
│ WATERSHED_CODE │     tothab_ha      │   FreshMarsh_pct_ha   │ SaltMarshNonDominant_pct_ha │ FreshwaterWoody_pct_ha │ DeepwaterFresh_pct_ha │ FreshShallowOpenWater_pct_ha │ MudflatSalt_pct_ha │
│    varchar     │       double       │        double         │           double            │         double         │        double         │            double            │       double       │
├────────────────┼────────────────────┼───────────────────────┼─────────────────────────────┼────────────────────────┼───────────────────────┼──────────────────────────────┼────────────────────┤
│ BF02347        │  9765.451241089459 │                  NULL │         0.06803493008312009 │                   NULL │                  NULL │                         NULL │               NULL │
│ BF02347        │  9765.

In [110]:
cols = con.sql('describe habpivot').df()['column_name'].tolist()
con.sql(f"""SELECT * FROM habpivot ORDER BY {hucidfld}""")
for i in con.sql("SELECT distinct(name) FROM crossnwi").fetchall():
    j = f"{i[0].replace('_','')}_pct_ha"
    if j not in cols:
        con.sql(f"ALTER TABLE habpivot ADD COLUMN {j} DOUBLE")
con.sql(f"""SELECT * FROM habpivot ORDER BY {hucidfld}""")

┌────────────────┬────────────────────┬───────────────────────┬─────────────────────────────┬────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────┐
│ WATERSHED_CODE │     tothab_ha      │   FreshMarsh_pct_ha   │ SaltMarshNonDominant_pct_ha │ FreshwaterWoody_pct_ha │ DeepwaterFresh_pct_ha │ FreshShallowOpenWater_pct_ha │ MudflatSalt_pct_ha │
│    varchar     │       double       │        double         │           double            │         double         │        double         │            double            │       double       │
├────────────────┼────────────────────┼───────────────────────┼─────────────────────────────┼────────────────────────┼───────────────────────┼──────────────────────────────┼────────────────────┤
│ BF02347        │  9765.451241089459 │                  NULL │         0.06803493008312009 │                   NULL │                  NULL │                         NULL │               NULL │
│ BF02347        │  9765.

In [111]:
con.sql(f"""CREATE OR REPLACE TABLE habbyhuc AS 
        SELECT {hucidfld}, tothab_ha,
        {', '.join([f"sum(coalesce({i},0)) AS {i}" for i in con.sql("DESCRIBE habpivot").df().column_name.to_list()[2:]])}
        FROM habpivot
        GROUP BY {hucidfld}, tothab_ha
        ORDER BY {hucidfld}
""")
con.sql(f"""SELECT * FROM habbyhuc""")

┌────────────────┬────────────────────┬───────────────────────┬─────────────────────────────┬────────────────────────┬────────────────────────┬──────────────────────────────┬────────────────────┐
│ WATERSHED_CODE │     tothab_ha      │   FreshMarsh_pct_ha   │ SaltMarshNonDominant_pct_ha │ FreshwaterWoody_pct_ha │ DeepwaterFresh_pct_ha  │ FreshShallowOpenWater_pct_ha │ MudflatSalt_pct_ha │
│    varchar     │       double       │        double         │           double            │         double         │         double         │            double            │       double       │
├────────────────┼────────────────────┼───────────────────────┼─────────────────────────────┼────────────────────────┼────────────────────────┼──────────────────────────────┼────────────────────┤
│ BF02347        │  9765.451241089459 │                   0.0 │         0.06803493008312009 │                    0.0 │                    0.0 │                          0.0 │  99.93196506991687 │
│ BF02348        │  

In [ ]:
#########
########
con.sql(f'''CREATE OR REPLACE TABLE athuclevel AS
SELECT *
FROM athuclevel
left join wtmeanbyhuc on athuclevel.{hucidfld}=wtmeanbyhuc.{hucidfld}
order by athuclevel.{hucidfld}
''')

In [116]:
con.sql(f'''CREATE OR REPLACE TABLE athuclevel AS
SELECT * EXCLUDE {hucidfld}_1
FROM athuclevel
LEFT JOIN habbyhuc on athuclevel.{hucidfld}=habbyhuc.{hucidfld}
order by athuclevel.{hucidfld}
''')

In [117]:
con.sql("DESCRIBE athuclevel")

┌──────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │ column_type │  null   │   key   │ default │  extra  │
│           varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ WATERSHED_CODE               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wshed_ha                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dud_lta                      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ demand_lta_kcal              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ popobj_lta                   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dud_80th                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ demand_80th_kcal             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ popobj_80th                  │ DOUBLE    

# Calculate protect/restore goals

In [118]:
con.sql(f'''CREATE OR REPLACE TABLE athuclevel AS
SELECT * EXCLUDE {hucidfld}_1,
CASE WHEN 
surpdef_lta_kcal < 0
THEN
abs(surpdef_lta_kcal/wtMean_kcal_per_ha)
ELSE 0
END
AS restoregoal_lta_ha,

CASE WHEN 
surpdef_80th_kcal < 0
THEN
abs(surpdef_80th_kcal/wtMean_kcal_per_ha)
ELSE 0
END
AS restoregoal_80th_ha,

CASE WHEN
wshed_ha - unavailha > 0
THEN
wshed_ha - unavailha
ELSE 0
END 
AS available_ha

FROM athuclevel
''')

In [119]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT * EXCLUDE (restoregoal_lta_ha, restoregoal_80th_ha),
CASE WHEN 
restoregoal_lta_ha > available_ha
THEN
available_ha
ELSE restoregoal_lta_ha
END
AS restoregoal_lta_ha,

CASE WHEN 
restoregoal_80th_ha > available_ha
THEN
available_ha
ELSE restoregoal_80th_ha
END
AS restoregoal_80th_ha,

FROM athuclevel
''')

In [120]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT *,
CASE WHEN 
nrgprot_lta_kcal > 0 
THEN
nrgprot_lta_kcal/wtMean_kcal_per_ha
ELSE 0
END
AS protectgoal_lta_ha,

CASE WHEN 
nrgprot_80th_kcal > 0
THEN
nrgprot_80th_kcal/wtMean_kcal_per_ha
ELSE 0
END
AS protectgoal_80th_ha,
FROM athuclevel
''')

In [121]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT * EXCLUDE (protectgoal_lta_ha, protectgoal_80th_ha, geometry),
CASE WHEN 
protectgoal_lta_ha > available_ha
THEN
available_ha
ELSE protectgoal_lta_ha
END
AS  protectgoal_lta_ha,

CASE WHEN 
protectgoal_80th_ha > available_ha
THEN
available_ha
ELSE protectgoal_80th_ha
END
AS protectgoal_80th_ha,

geometry        
FROM athuclevel
''')

# Save output to file

In [ ]:
'''
Protected wetlands, urban wetlands, and wetland energy all calculated by huc12.  Need to calculate total urban outside of
wetland energy

Calculations:
    Energy supply
        Total habitat energy within huc - THabNrg
        Total habitat hectares within huc - THabHA

    Energy demand
        LTA and X80 DUD by huc - TLTADUD anc X80DUD
        LTA and X80 Demand by huc - TLTADemand and X80Demand
        LTA and X80 Population objective by huc - LTAPopObj and X80PopObj
        
    Protected lands
        Total protected hectares by huc - ProtHA

    Protected habitat hectares and energy
        Total protected hectares - ProtHabHA
        Total protected energy - ProtHabNrg

    Weighted mean and calculations based off of it
        Weighted mean kcal/ha with weight being Total habitat energy
        Energy Protection needed - NrgProtRq
        Restoration HA based off of weighted mean - RstorHA
        Protection HA based off weighted mean - RstorProtHA  

'''
#################################
#################################
#################################


In [122]:
con.sql("DESCRIBE athuclevel").df().column_name.to_list()
con.sql("SELECT * EXCLUDE geometry FROM athuclevel")

┌────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬──────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬───────

In [123]:
out_f = r"D:\ABDUBounds\model_output\abdu_cn_east_v4.parquet"
con.sql(f"COPY (SELECT * FROM athuclevel) TO '{out_f}' (FORMAT PARQUET)")

### others/scratch

In [87]:
con.sql("show tables")

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ athuclevel    │
│ cn_prov       │
│ crossnwi      │
│ demand        │
│ habbyhuc      │
│ habpivot      │
│ hucdemand     │
│ hucs          │
│ protected     │
│ protwetlands  │
│ rdydemand     │
│ spp_demand    │
│ unavailable   │
│ urban         │
│ urbanwetlands │
│ wetlands      │
│ wtmean        │
│ wtmeanbyhuc   │
│ wtmeanpivot   │
├───────────────┤
│    19 rows    │
└───────────────┘

In [19]:
out_f = r"D:\ABDUBounds\model_output\hucdemand.parquet"
# con.sql(f"COPY (SELECT * FROM hucdemand) TO '{out_f}' (FORMAT PARQUET)")
sql = f"""CREATE OR REPLACE TABLE hucdemand AS
SELECT * FROM read_parquet('{out_f}')
"""
con.execute(sql)
con.sql("DESCRIBE hucdemand")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ species        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LTADUD         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LTADemand      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LTAPopObj      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ x80DUD         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ X80Demand      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ X80PopObj      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [20]:
out_f = r"D:\ABDUBounds\model_output\hucs.parquet"
# con.sql(f"COPY (SELECT * FROM hucs) TO '{out_f}' (FORMAT PARQUET)")
sql = f"""CREATE OR REPLACE TABLE hucs AS
SELECT * FROM read_parquet('{out_f}')
"""
con.execute(sql)
con.sql("DESCRIBE hucs")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ MERGE_SRC      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PROV           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [23]:
out_f = r"D:\ABDUBounds\model_output\urbanwetlands.parquet"
# con.sql(f"COPY (SELECT * FROM urbanwetlands) TO '{out_f}' (FORMAT PARQUET)")
sql = f"""CREATE OR REPLACE TABLE urbanwetlands AS
SELECT * FROM read_parquet('{out_f}')
"""
con.execute(sql)
con.sql("DESCRIBE urbanwetlands")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ geometry       │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
│ name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WATERSHED_CODE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ha             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ kcal           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ urbanNrgy      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘